# 직접 해 보기 2 — 파이썬 코드로 따라 해 보기

이 장의 자료는 AI에게 보여 줄 수 없으므로, 분석은 내 컴퓨터에서 내 손으로 돌려야 한다. AI가 코드를 짜 주더라도 그 코드를 읽고, 돌리고, 결과가 맞는지 확인하는 것은 내 몫이다.

이 노트북은 이 장의 보고서와 본문의 표를 파이썬으로 직접 만든다. 이 장에서 새로 배운 두 가지, 곧 가중치로 숫자를 바로잡는 것과 표본설계대로 표준오차를 계산하는 것을 코드로 보여 주는 것이 중심이다.

1. 자료 읽기 — 내려받은 SAS 파일을 읽는다
2. 분석 대상과 변수 만들기
3. 가중 평균 — 표본과 모집단의 생김새(2절)
4. 표본설계를 반영한 회귀분석 함수
5. 보고서의 회귀식
6. 표준오차를 세 가지로 계산하기(표 2-1)
7. 변수를 하나씩 더하기(표 3-1)
8. 나이 곡선 — 예상 혈압 계산
9. 허리둘레와 상호작용(표 4-1, 표 4-3)
10. 혈압약을 다루는 네 방법(표 5-1)
11. 보고서의 그림 왼쪽 패널

**주의.** 이 노트북은 한 사람 한 사람의 기록을 화면에 출력하지 않는다. `df.head()`처럼 줄을 보여 주는 코드를 더하지 않는다. 출력이 노트북 파일에 저장되기 때문이다. 이 노트북을 남에게 줄 때는 출력을 지우고(Kernel → Restart & Clear Output) 준다.

## 준비 — 패키지와 글꼴

`import`로 쓸 패키지를 불러온다. `patsy`는 회귀식 문자열(`'혈압 ~ 여성 + 나이'`)을 계산할 수 있는 표로 바꿔 주는 패키지로, statsmodels를 설치하면 함께 들어온다. `scipy.stats`는 $t$ 분포로 $p$값과 신뢰구간을 계산할 때 쓴다. 모두 Anaconda와 Colab에 들어 있다.

그림의 한글 글꼴은 다른 장과 같다. 윈도우는 맑은 고딕(`Malgun Gothic`), 맥은 애플고딕(`AppleGothic`)이다.

In [ ]:
import numpy as np                       # 수치 계산
import pandas as pd                      # 표(데이터프레임) 다루기
import patsy                             # 회귀식 문자열 → 계산용 표
from scipy import stats                  # t 분포
import matplotlib.pyplot as plt          # 그림

plt.rcParams['font.family'] = 'Malgun Gothic'   # 윈도우 (맥은 'AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

## 1. 자료 읽기

국민건강영양조사 누리집에서 이행서약을 거쳐 내려받은 2024년 기본DB의 SAS 파일을 읽는다(직접 해 보기 1의 "준비할 것"). 압축을 푼 폴더를 아래 `DATA_DIR =` 줄에, 파일 이름 `hn24_all.sas7bdat`를 `SAS =` 줄에 적는다. 자료는 노트북과 다른 폴더에 두고 경로로 불러들인다. 그래서 노트북 파일을 옮기거나 남에게 주어도 자료 파일은 따라가지 않는다.

`Path`는 폴더 경로를 다루는 도구다. `DATA_DIR / SAS`처럼 `/`로 폴더와 파일 이름을 이어 붙이면 파일의 전체 경로가 된다. 따옴표 앞의 `r`은 윈도우 경로의 역슬래시(`\`)를 글자 그대로 읽으라는 표시다. 폴더 경로는 내 컴퓨터에 맞게 바꿔 적는다. 맥이라면 `Path('/Users/이름/knhanes/all24')`처럼 쓴다.

`pd.read_sas`가 SAS 파일을 읽는다. 한글 라벨이 있어 `encoding='cp949'`를 준다. 파일이 커서 읽는 데 조금 걸린다. 읽은 뒤에는 사람 수와 변수 수만 출력한다.

Colab에 올리는 것은 외부 서비스에 자료를 올리는 일이다. 이 자료는 내 컴퓨터의 주피터에서 연다(직접 해 보기 1의 "먼저 확인할 것").

In [ ]:
from pathlib import Path

DATA_DIR = Path(r'C:\Work\Books\book5\data\knhanes\all24')
SAS = 'hn24_all.sas7bdat'          # 내려받은 파일 이름
raw = pd.read_sas(DATA_DIR / SAS, encoding='cp949')
print(raw.shape)                   # (사람 수, 변수 수)만 본다

## 2. 분석 대상과 변수 만들기

보고서의 분석 대상은 "19세 이상, 임신부와 학력·혈압 결측 제외, 혈압약 복용자 포함"이다. 가중치가 0인 사람(검진에 참여하지 않은 사람)도 뺀다.

| 새 변수 | 원 변수와 만드는 법 |
|---|---|
| `학력` | `edu`: 1 초졸 이하, 2 중졸, 3 고졸, 4 대졸 이상 |
| `여성` | `sex`가 2이면 1, 아니면 0 |
| `나이` | `age` 그대로(80세 이상은 80으로 기록돼 있다) |
| `혈압` | `HE_sbp`, 수축기 혈압(mmHg) |
| `혈압약` | `DI1_2`가 1~4(혈압조절제 복용)이면 1 |
| `허리` | `HE_wc`, 허리둘레(cm) |
| `w`, `층`, `군집` | 가중치 `wt_itvex`, 층 `kstrata`, 군집 `psu` |

`pd.Categorical`은 학력을 "초졸 이하, 중졸, 고졸, 대졸 이상"의 순서가 있는 범주로 만든다. 이 순서의 첫째(초졸 이하)가 회귀식에서 기준이 된다.

In [ ]:
EDU = {1: '초졸 이하', 2: '중졸', 3: '고졸', 4: '대졸 이상'}

d = raw[(raw['age'] >= 19) & (raw['HE_prg'] != 1) & (raw['wt_itvex'] > 0)].copy()
d['학력'] = pd.Categorical(d['edu'].map(EDU), categories=list(EDU.values()))
d['여성'] = (d['sex'] == 2).astype(int)
d['나이'] = d['age']
d['혈압'] = d['HE_sbp']
d['혈압약'] = d['DI1_2'].isin([1, 2, 3, 4]).astype(int)
d['허리'] = d['HE_wc']
d['w'] = d['wt_itvex']
d['층'] = d['kstrata']
d['군집'] = d['psu']

d = d.dropna(subset=['학력', '혈압'])          # 학력·혈압 결측 제외
print('분석 대상', len(d), '명')               # 보고서: 5,731명

## 3. 가중 평균 — 표본과 모집단의 생김새

가중 평균의 원리는 한 줄이다. 사람마다 값에 가중치를 곱해 더하고, 가중치의 합으로 나눈다(2절). `np.average(값, weights=가중치)`가 이 계산을 한다.

표본에는 고령자와 여성이 실제보다 많이 들어 있어서, 가중치를 쓰면 평균 나이가 내려가고 여성 비율도 내려간다. 평균 혈압과 혈압약 복용 비율도 함께 본다.

In [ ]:
for col in ['나이', '여성', '혈압', '혈압약']:
    plain = d[col].mean()
    weighted = np.average(d[col], weights=d['w'])
    print(f'{col:4s}  그냥 낸 평균 {plain:7.3f}   가중 평균 {weighted:7.3f}')

print('유효 표본 크기', round(d['w'].sum()**2 / (d['w']**2).sum()))   # 가중치가 고르지 않은 만큼 줄어든다

## 4. 표본설계를 반영한 회귀분석 함수

계수는 가중 최소제곱으로 구한다. 잔차제곱에 가중치를 곱해 더한 합이 가장 작은 직선이다. 표준오차는 사람을 뽑은 방식대로 계산한다(2절). 계산의 뼈대는 이렇다.

1. 사람마다 "설명변수 × 가중치 × 잔차"를 구한다. 계수가 표본에 따라 흔들리는 것은 이 값들이 흔들리기 때문이다.
2. 이 값을 **군집(동네)마다 더한다**. 같은 동네 사람은 닮아서 함께 움직이니, 한 덩어리로 센다.
3. **층(칸)마다** 군집 합들이 그 층의 평균에서 얼마나 흩어져 있는지를 잰다. 층끼리의 차이는 빼고 층 안의 흩어짐만 쓴다.
4. 이것을 다 더해 계수의 분산을 만들고, 제곱근이 표준오차다. $p$값과 신뢰구간은 자유도(군집 수 − 층 수)의 $t$ 분포로 계산한다.

이 방식을 테일러 선형화라고 부르고, Stata의 `svy`, R의 survey 패키지, SAS의 `PROC SURVEYREG`가 기본으로 쓰는 계산과 같다. 층을 하나로 두고 군집을 사람마다 따로 두면 "제비뽑기로 뽑았다"고 보는 계산이 된다(6절).

In [ ]:
def svy_reg(formula, data, weight='w', strata='층', cluster='군집'):
    y, X = patsy.dmatrices(formula, data, return_type='dataframe')   # 결측이 있는 줄은 빠진다
    dd = data.loc[X.index]
    w = dd[weight].to_numpy()
    Xv, yv = X.to_numpy(), y.to_numpy().ravel()

    bread = np.linalg.inv(Xv.T @ (Xv * w[:, None]))
    b = bread @ (Xv.T @ (w * yv))                     # 가중 최소제곱 계수
    e = yv - Xv @ b                                    # 잔차

    u = pd.DataFrame(Xv * (w * e)[:, None], index=X.index)       # 1. 사람마다
    z = u.groupby([dd[strata], dd[cluster]]).sum()               # 2. 군집마다 더한다
    meat = np.zeros((Xv.shape[1], Xv.shape[1]))
    for _, zh in z.groupby(level=0):                             # 3. 층마다
        nh = len(zh)
        if nh > 1:
            c = zh.to_numpy() - zh.to_numpy().mean(axis=0)
            meat += nh / (nh - 1) * c.T @ c
    V = bread @ meat @ bread                                     # 4. 계수의 분산
    se = np.sqrt(np.diag(V))

    df = dd[cluster].nunique() - dd[strata].nunique()            # 자유도
    t = b / se
    q = stats.t.ppf(0.975, df)
    r2 = 1 - np.sum(w * e**2) / np.sum(w * (yv - np.average(yv, weights=w))**2)
    out = pd.DataFrame({'계수': b, '표준오차': se, 't값': t,
                        '하한': b - q * se, '상한': b + q * se,
                        'p값': 2 * stats.t.sf(abs(t), df)}, index=X.columns)
    out.attrs = {'n': len(yv), 'R2': r2, 'df': df}
    return out

## 5. 보고서의 회귀식

보고서의 회귀식을 그대로 옮긴다.

수축기 혈압 = 학력 더미 3개(기준: 초졸 이하) + 여성 + 나이 + 나이²

`C(학력)`은 학력을 더미 변수로 넣으라는 뜻이고(범주의 첫째가 기준), `I(나이**2)`는 나이의 제곱을 회귀식 안에서 계산해 넣는다. 결과가 보고서의 표와 같은지 대조한다. 대졸 이상 −7.53(0.90), 여성 −6.29(0.40), $n$ = 5,731, $R^2$ = 0.212다.

In [ ]:
F = '혈압 ~ C(학력) + 여성 + 나이 + I(나이**2)'
r = svy_reg(F, d)
print(r.round(4))
print(r.attrs)

## 6. 표준오차를 세 가지로 계산하기

2절의 표 2-1이다. 같은 회귀식을 세 가지로 계산해, 가중치는 계수를 바꾸고 표본설계는 표준오차만 바꾸는 것을 확인한다.

- 가중치 없음, 제비뽑기 가정: 가중치를 모두 1로, 층을 하나로, 군집을 사람마다 따로
- 가중치 적용, 제비뽑기 가정: 가중치만 쓴다
- 가중치 적용, 층과 군집 반영: 보고서의 계산

In [ ]:
d['하나'] = 1                       # 층이 하나뿐
d['한사람'] = np.arange(len(d))      # 군집 = 사람마다 따로
d['가중없음'] = 1.0

ways = {'가중치 없음, 제비뽑기 가정': dict(weight='가중없음', strata='하나', cluster='한사람'),
        '가중치 적용, 제비뽑기 가정': dict(strata='하나', cluster='한사람'),
        '가중치 적용, 층과 군집 반영': dict()}
rows = {}
for name, opt in ways.items():
    rk = svy_reg(F, d, **opt)
    rows[name] = {'대졸 이상': f"{rk.loc['C(학력)[T.대졸 이상]', '계수']:.2f} ({rk.loc['C(학력)[T.대졸 이상]', '표준오차']:.3f})",
                  '여성': f"{rk.loc['여성', '계수']:.2f} ({rk.loc['여성', '표준오차']:.3f})"}
pd.DataFrame(rows).T

## 7. 변수를 하나씩 더하기

3절의 표 3-1이다. 학력만 넣은 회귀식에서 출발해 여성, 나이와 나이²를 더해 가며 대졸 이상의 계수를 본다. 학력만 넣으면 −15.0, 성별과 나이를 같게 맞추면 −7.5다. 대졸 이상에는 젊은 사람이 많아서, 그냥 비교하면 나이의 차이까지 함께 비교하게 되기 때문이다.

In [ ]:
steps = {'학력만': '혈압 ~ C(학력)',
         '+ 여성': '혈압 ~ C(학력) + 여성',
         '+ 나이, 나이²': F}
for name, f in steps.items():
    rk = svy_reg(f, d)
    print(f"{name:10s} 대졸 이상 {rk.loc['C(학력)[T.대졸 이상]', '계수']:7.2f}   R² {rk.attrs['R2']:.3f}")

print('학력별 평균 나이(가중):')
print(d.groupby('학력', observed=True).apply(lambda g: np.average(g['나이'], weights=g['w']), include_groups=False).round(1))

## 8. 나이 곡선 — 예상 혈압 계산

나이와 나이²의 계수는 하나씩 읽지 않고 함께 곡선 하나로 읽는다(3절). 회귀식에 값을 넣어 예상 혈압을 계산해 본다. 50세 초졸 이하 남성은 127.3, 70세는 134.0이다.

In [ ]:
b = r['계수']
for age in [30, 50, 70]:
    pred = b['Intercept'] + b['나이'] * age + b['I(나이 ** 2)'] * age**2   # 초졸 이하, 남성
    print(age, '세', round(pred, 1))

## 9. 허리둘레와 상호작용

4절의 두 표다.

- **허리둘레**(표 4-1): 허리둘레가 있는 사람만 남겨, 보고서의 회귀식과 허리둘레를 더한 회귀식을 같은 사람으로 추정한다. 대졸 이상의 계수가 −7.5에서 −5.7로 바뀐다.
- **상호작용**(표 4-3): `C(학력):여성`을 더하면 학력의 계수가 남녀에게 따로 나온다. 남성의 대졸 이상은 대졸 이상의 계수 그대로, 여성은 거기에 상호작용 계수를 더한 값이다.

In [ ]:
w4 = d.dropna(subset=['허리'])
for name, f in [('보고서의 회귀식', F), ('+ 허리둘레', F + ' + 허리')]:
    rk = svy_reg(f, w4)
    print(f"{name:10s} 대졸 이상 {rk.loc['C(학력)[T.대졸 이상]', '계수']:6.2f}   n {rk.attrs['n']}")

ri = svy_reg(F + ' + C(학력):여성', d)
male = ri.loc['C(학력)[T.대졸 이상]', '계수']
inter = ri.loc['C(학력)[T.대졸 이상]:여성', '계수']
print('대졸 이상 — 남성', round(male, 2), ' 여성', round(male + inter, 2), ' 상호작용', round(inter, 2))

## 10. 혈압약을 다루는 네 방법

5절의 표 5-1이다. 혈압약을 먹는 사람을 그대로 넣기, 빼기, 혈압에 10이나 15를 더하기의 네 방법으로 대졸 이상의 계수를 비교한다. 방법에 따라 −7.2에서 −9.6까지 달라지지만, 학력이 높을수록 혈압이 낮다는 결론은 같다(강건성 점검).

In [ ]:
cases = {'그대로 넣기 (보고서)': (d, 0),
         '약을 먹는 사람을 빼기': (d[d['혈압약'] == 0], 0),
         '혈압에 10을 더하기': (d, 10),
         '혈압에 15를 더하기': (d, 15)}
for name, (data, add) in cases.items():
    dk = data.copy()
    dk['혈압'] = dk['혈압'] + add * dk['혈압약']
    rk = svy_reg(F, dk)
    print(f"{name:14s} 대졸 이상 {rk.loc['C(학력)[T.대졸 이상]', '계수']:6.2f} ({rk.loc['C(학력)[T.대졸 이상]', '표준오차']:.2f})   n {rk.attrs['n']}")

## 11. 보고서의 그림 왼쪽 패널

학력별 가중 평균 수축기 혈압을 남녀로 나눠 그린다. 나이를 맞추지 않은 그냥 비교라서 학력에 따른 차이가 크게 보인다. 이 그림의 차이가 회귀식에서 절반쯤으로 줄어드는 까닭이 3절의 이야기다.

In [ ]:
avg = d.groupby(['여성', '학력'], observed=True).apply(lambda g: np.average(g['혈압'], weights=g['w']), include_groups=False)

plt.figure(figsize=(5.5, 4))
for sex, name, col in [(0, '남성', '#2a78d6'), (1, '여성', '#eb6834')]:
    plt.plot(list(EDU.values()), avg[sex].to_numpy(), marker='o', color=col, label=name)
plt.ylabel('평균 수축기 혈압 (mmHg, 가중)')
plt.legend()
plt.show()

## 정리

이 노트북의 코드는 보고서의 회귀 결과(5절)와 본문의 표 다섯 개(6, 7, 9, 10절)를 직접 만들었다. 새로 쓴 것은 4절의 함수 하나다. 가중치로 계수를 구하고, 군집과 층을 따라 표준오차를 계산한다. 나머지는 그 함수에 회귀식과 자료를 바꿔 넣은 것뿐이다.

AI에게 코드를 받았다면, 그 코드의 결과가 이 노트북의 결과와 같은지 맞춰 보는 것이 가장 확실한 점검이다. 그리고 노트북을 남에게 주기 전에는 출력을 지운다.